<a href="https://www.kaggle.com/code/nmavros/deepfire-forecaster?scriptVersionId=300757052" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import os
import glob
import torch
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split

print("Initializing environment and defining the Dataset class...")

# Set random seed for reproducibility in research
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    
set_seed(42)

class SatFireDataset(Dataset):
    def __init__(self, event_dirs, target_max=100.0):
        self.event_dirs = event_dirs
        self.target_max = target_max

    def __len__(self):
        return len(self.event_dirs)

    def __getitem__(self, idx):
        event_path = self.event_dirs[idx]
        
        # Load VIIRS Thermal Sequence (Days 1 to 3)
        viirs_files = sorted(glob.glob(os.path.join(event_path, "VIIRS", "*.tif")))
        viirs_seq = []
        for f in viirs_files[:3]: 
            with rasterio.open(f) as src:
                viirs_seq.append(src.read())
        viirs_seq = np.array(viirs_seq) 
        
        # Load LULC (Land Use / Land Cover)
        lulc_file = glob.glob(os.path.join(event_path, "LULC", "*.tif"))[0]
        with rasterio.open(lulc_file) as src:
            lulc = src.read()
            
        # Load Target (FirePred Day 4)
        target_file = glob.glob(os.path.join(event_path, "FirePred", "*.tif"))[0]
        with rasterio.open(target_file) as src:
            target = src.read(4) 
            
        # Data Normalization & Signal Clipping
        target = np.nan_to_num(target, nan=0.0)
        viirs_seq = np.nan_to_num(viirs_seq, nan=0.0)
        lulc = np.nan_to_num(lulc, nan=0.0)
        
        # Clip outliers and apply Min-Max Scaling [0, 1]
        target = np.clip(target, a_min=0.0, a_max=self.target_max)
        target = target / self.target_max
        
        viirs_tensor = torch.tensor(viirs_seq, dtype=torch.float32)
        lulc_tensor = torch.tensor(lulc, dtype=torch.float32)
        target_tensor = torch.tensor(target, dtype=torch.float32).unsqueeze(0) 
        
        return viirs_tensor, lulc_tensor, target_tensor

print("Environment setup complete. Dataset class is ready.")

Initializing environment and defining the Dataset class...
Environment setup complete. Dataset class is ready.


In [5]:
import os
import glob
import torch
import numpy as np
import rasterio
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

print("Initializing environment with Spatial Interpolation...")

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    
set_seed(42)

class SatFireDataset(Dataset):
    def __init__(self, event_dirs, target_max=100.0, spatial_size=256):
        self.event_dirs = event_dirs
        self.target_max = target_max
        self.spatial_size = (spatial_size, spatial_size)

    def __len__(self):
        return len(self.event_dirs)

    def __getitem__(self, idx):
        event_path = self.event_dirs[idx]
        
        # Load VIIRS Day Sequence (Days 1 to 3)
        viirs_files = sorted(glob.glob(os.path.join(event_path, "VIIRS_Day", "*.tif")))
        viirs_seq = []
        for f in viirs_files[:3]: 
            with rasterio.open(f) as src:
                viirs_seq.append(src.read())
        viirs_seq = np.array(viirs_seq) 
        
        # Load ESRI LULC 
        lulc_file = glob.glob(os.path.join(event_path, "ESRI_LULC", "*.tif"))[0]
        with rasterio.open(lulc_file) as src:
            lulc = src.read()
            
        # Load Target FirePred (Day 4)
        target_file = glob.glob(os.path.join(event_path, "FirePred", "*.tif"))[0]
        with rasterio.open(target_file) as src:
            target = src.read(4) 
            
        # Apply normalization and clipping
        target = np.nan_to_num(target, nan=0.0)
        viirs_seq = np.nan_to_num(viirs_seq, nan=0.0)
        lulc = np.nan_to_num(lulc, nan=0.0)
        
        target = np.clip(target, a_min=0.0, a_max=self.target_max)
        target = target / self.target_max
        
        # Convert to Tensors
        viirs_tensor = torch.tensor(viirs_seq, dtype=torch.float32) # Shape: [3, 8, H, W]
        lulc_tensor = torch.tensor(lulc, dtype=torch.float32)       # Shape: [1, H, W]
        target_tensor = torch.tensor(target, dtype=torch.float32).unsqueeze(0) # Shape: [1, H, W]
        
        # --- SPATIAL RESIZING BLOCK ---
        # Reshape VIIRS for interpolation: [1, 24, H, W]
        _, _, current_H, current_W = viirs_tensor.shape
        viirs_flat = viirs_tensor.view(1, 24, current_H, current_W)
        
        # Bilinear for continuous features (VIIRS and Target)
        viirs_resized = F.interpolate(viirs_flat, size=self.spatial_size, mode='bilinear', align_corners=False)
        viirs_tensor = viirs_resized.view(3, 8, self.spatial_size[0], self.spatial_size[1])
        
        target_resized = F.interpolate(target_tensor.unsqueeze(0), size=self.spatial_size, mode='bilinear', align_corners=False)
        target_tensor = target_resized.squeeze(0)
        
        # Nearest for categorical LULC labels
        lulc_resized = F.interpolate(lulc_tensor.unsqueeze(0), size=self.spatial_size, mode='nearest')
        lulc_tensor = lulc_resized.squeeze(0)
        
        return viirs_tensor, lulc_tensor, target_tensor

print("Dataset class updated with Spatial Resizing.")
print("Scanning dataset directories for integrity...")

KAGGLE_DATA_ROOT = "/kaggle/input/ts-satfire/ts-satfire"
valid_event_dirs = []
all_dirs = sorted([
    os.path.join(KAGGLE_DATA_ROOT, d) for d in os.listdir(KAGGLE_DATA_ROOT) 
    if os.path.isdir(os.path.join(KAGGLE_DATA_ROOT, d))
])

for directory in all_dirs:
    viirs_files = glob.glob(os.path.join(directory, "VIIRS_Day", "*.tif"))
    lulc_files = glob.glob(os.path.join(directory, "ESRI_LULC", "*.tif"))
    target_files = glob.glob(os.path.join(directory, "FirePred", "*.tif"))
    
    if len(viirs_files) >= 3 and len(lulc_files) >= 1 and len(target_files) >= 1:
        valid_event_dirs.append(directory)

total_events = len(valid_event_dirs)
print(f"Valid fire events found: {total_events}")

train_size = int(0.7 * total_events)
val_size = int(0.15 * total_events)
test_size = total_events - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_dirs, val_dirs, test_dirs = random_split(
    valid_event_dirs, 
    [train_size, val_size, test_size], 
    generator=generator
)

# Instantiate the Dataset with resizing target configuration
TARGET_MAX = 100.0
SPATIAL_RESOLUTION = 256
train_dataset = SatFireDataset(train_dirs, target_max=TARGET_MAX, spatial_size=SPATIAL_RESOLUTION)
val_dataset = SatFireDataset(val_dirs, target_max=TARGET_MAX, spatial_size=SPATIAL_RESOLUTION)
test_dataset = SatFireDataset(test_dirs, target_max=TARGET_MAX, spatial_size=SPATIAL_RESOLUTION)

# Create the DataLoaders with a safe batch size
BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

print("DataLoaders successfully recreated with unified dimensions.")

Initializing environment with Spatial Interpolation...
Dataset class updated with Spatial Resizing.
Scanning dataset directories for integrity...
Valid fire events found: 150
DataLoaders successfully recreated with unified dimensions.


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("Initializing the Hybrid CNN-Transformer Architecture...")

class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class SpatialEncoder(nn.Module):
    def __init__(self, viirs_channels=8, lulc_channels=1, embed_dim=128):
        super().__init__()
        in_channels = viirs_channels + lulc_channels
        self.layer1 = CNNBlock(in_channels, 32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.layer2 = CNNBlock(32, 64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.layer3 = CNNBlock(64, embed_dim)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, viirs_seq, lulc):
        B, T, C_v, H, W = viirs_seq.shape
        C_l = lulc.shape[1]
        
        viirs_reshaped = viirs_seq.view(B * T, C_v, H, W)
        lulc_repeated = lulc.unsqueeze(1).repeat(1, T, 1, 1, 1)
        lulc_reshaped = lulc_repeated.view(B * T, C_l, H, W)
        
        x = torch.cat([viirs_reshaped, lulc_reshaped], dim=1)
        
        x = self.pool1(self.layer1(x))
        x = self.pool2(self.layer2(x))
        features = self.pool3(self.layer3(x))
        
        _, C_f, H_f, W_f = features.shape
        return features.view(B, T, C_f, H_f, W_f)

class SimpleSpatiotemporalTransformer(nn.Module):
    def __init__(self, embed_dim=128, num_heads=4, num_layers=2, seq_len=3, spatial_size=32):
        super().__init__()
        self.num_tokens = seq_len * spatial_size * spatial_size
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_tokens, embed_dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 2,
            dropout=0.1, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, features_temporal):
        B, T, C, H, W = features_temporal.shape
        x = features_temporal.view(B, T * H * W, C)
        x = x + self.pos_embed
        x = self.transformer(x)
        return x.view(B, T, C, H, W)

class TemporalAttentionPooling(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.attention_net = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 2, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 2, 1, kernel_size=1)
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x_reshaped = x.view(B * T, C, H, W)
        scores = self.attention_net(x_reshaped).view(B, T, -1).mean(dim=2)
        weights = F.softmax(scores, dim=1)
        weights_expanded = weights.view(B, T, 1, 1, 1)
        pooled_features = torch.sum(x * weights_expanded, dim=1)
        return pooled_features, weights

class GenerativeDecoder(nn.Module):
    def __init__(self, in_channels=128):
        super().__init__()
        self.up1 = nn.Sequential(nn.ConvTranspose2d(in_channels, 64, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.up2 = nn.Sequential(nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.up3 = nn.Sequential(nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True))
        self.final_conv = nn.Conv2d(16, 1, kernel_size=3, padding=1)
        
    def forward(self, x):
        return self.final_conv(self.up3(self.up2(self.up1(x))))

class SatGenTransformerNormalized(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        self.spatial_encoder = SpatialEncoder(embed_dim=embed_dim)
        self.transformer = SimpleSpatiotemporalTransformer(embed_dim=embed_dim)
        self.attention_pool = TemporalAttentionPooling(in_channels=embed_dim)
        self.decoder = GenerativeDecoder(in_channels=embed_dim)

    def forward(self, viirs_seq, lulc):
        features = self.spatial_encoder(viirs_seq, lulc)
        attended_features = self.transformer(features)
        pooled_features, attn_weights = self.attention_pool(attended_features)
        
        # Raw linear output
        raw_predictions = self.decoder(pooled_features)
        
        # Sigmoid activation to strictly bound outputs between 0.0 and 1.0
        predictions = torch.sigmoid(raw_predictions)
        
        return predictions, attn_weights

class MaskedMSELoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, predictions, targets):
        # We apply MSE directly on the normalized [0, 1] values
        loss = F.mse_loss(predictions, targets)
        return loss

print("Architecture and MaskedMSELoss loaded successfully. Ready for training.")

Initializing the Hybrid CNN-Transformer Architecture...
Architecture and MaskedMSELoss loaded successfully. Ready for training.


In [ ]:
import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.display import display, FileLink

print("============================================================")
print("Initializing Final Normalized Training Engine")
print("============================================================\n")

# --- 1. SETUP ---
EPOCHS = 10
EMBED_DIM = 128
SAVE_PATH = "novhyt_final_best.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SatGenTransformerNormalized(embed_dim=EMBED_DIM).to(device)
criterion = MaskedMSELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

train_losses = []
val_losses = []
best_val_loss = float('inf')

print(f"Starting normalized training for {EPOCHS} epochs on {device}...\n")

# --- 2. TRAINING LOOP ---
for epoch in range(EPOCHS):
    model.train()
    epoch_train_loss = 0.0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    
    for viirs_batch, lulc_batch, target_batch in train_pbar:
        viirs_batch = viirs_batch.to(device)
        lulc_batch = lulc_batch.to(device)
        target_batch = target_batch.to(device)
        
        optimizer.zero_grad()
        predictions, _ = model(viirs_batch, lulc_batch)
        loss = criterion(predictions, target_batch)
        loss.backward()
        optimizer.step()
        
        epoch_train_loss += loss.item()
        # You will now see tiny numbers here, bounded between 0 and 1
        train_pbar.set_postfix({'mse': f"{loss.item():.6f}"})
        
    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Valid]")
        for viirs_batch, lulc_batch, target_batch in val_pbar:
            viirs_batch = viirs_batch.to(device)
            lulc_batch = lulc_batch.to(device)
            target_batch = target_batch.to(device)
            
            predictions, _ = model(viirs_batch, lulc_batch)
            loss = criterion(predictions, target_batch)
            
            epoch_val_loss += loss.item()
            val_pbar.set_postfix({'mse': f"{loss.item():.6f}"})
            
    avg_val_loss = epoch_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    print("\n" + "-"*50)
    print(f"EPOCH {epoch+1} STATISTICAL REPORT")
    print(f"Average Training Normalized MSE   : {avg_train_loss:.6f}")
    print(f"Average Validation Normalized MSE : {avg_val_loss:.6f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        print("Status: Validation MSE improved. Model parameters saved.")
    else:
        print("Status: No improvement in Validation MSE.")
    print("-" * 50 + "\n")

print("Training Complete! Generating Paper Visualizations...\n")

# --- 3. LEARNING CURVE PLOT ---
plt.figure(figsize=(10, 6))
plt.plot(range(1, EPOCHS+1), train_losses, label='Train Normalized MSE', marker='o', color='blue')
plt.plot(range(1, EPOCHS+1), val_losses, label='Validation Normalized MSE', marker='s', color='orange')
plt.title('Learning Curve: Normalized Mean Squared Error')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss [0, 1]')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
learning_curve_file = 'learning_curve_final.png'
plt.savefig(learning_curve_file, bbox_inches='tight', dpi=300)
plt.close()

# --- 4. INFERENCE & ERROR MAP (DENORMALIZED) ---
print("Running inference and denormalizing outputs for visualization...")

model.load_state_dict(torch.load(SAVE_PATH))
model.eval()

viirs_batch, lulc_batch, target_batch = next(iter(test_loader))

with torch.no_grad():
    predictions, _ = model(viirs_batch[0:1].to(device), lulc_batch[0:1].to(device))
    pred_map = predictions.cpu()[0, 0]

target_clean = torch.nan_to_num(target_batch[0, 0], nan=0.0)

# Denormalize back to physical units for the final paper plots
TARGET_MAX = 100.0 
target_physical = target_clean * TARGET_MAX
pred_physical = pred_map * TARGET_MAX
error_map = torch.abs(pred_physical - target_physical).numpy()

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Spatiotemporal Fire Regression: Final Output', fontsize=18)

for i in range(3):
    axes[0, i].imshow(viirs_batch[0, i, 0].cpu().numpy(), cmap='viridis')
    axes[0, i].set_title(f"Input Day {i+1} (Thermal Sequence)")
    axes[0, i].axis('off')

vmin = min(target_physical.min().item(), pred_physical.min().item())
vmax = max(target_physical.max().item(), pred_physical.max().item())

im1 = axes[1, 0].imshow(target_physical.numpy(), cmap='inferno', vmin=vmin, vmax=vmax)
axes[1, 0].set_title("Actual Fire Truth (Physical Units)")
axes[1, 0].axis('off')
fig.colorbar(im1, ax=axes[1, 0], fraction=0.046, pad=0.04)

im2 = axes[1, 1].imshow(pred_physical.numpy(), cmap='inferno', vmin=vmin, vmax=vmax)
axes[1, 1].set_title("Model Prediction (Physical Units)")
axes[1, 1].axis('off')
fig.colorbar(im2, ax=axes[1, 1], fraction=0.046, pad=0.04)

im3 = axes[1, 2].imshow(error_map, cmap='hot')
axes[1, 2].set_title("Absolute Error Map (Black=Perfect)")
axes[1, 2].axis('off')
fig.colorbar(im3, ax=axes[1, 2], fraction=0.046, pad=0.04)

plt.tight_layout()
eval_maps_file = 'final_evaluation_maps.png'
plt.savefig(eval_maps_file, bbox_inches='tight', dpi=300)
plt.close()

# --- 5. GENERATE DOWNLOAD LINKS ---
print("============================================================")
print("DOWNLOAD READY - Click the links below to save to your PC:")
print("============================================================")
display(FileLink(SAVE_PATH))
display(FileLink(learning_curve_file))
display(FileLink(eval_maps_file))

Initializing Final Normalized Training Engine

Starting normalized training for 10 epochs on cuda...



Epoch 1/10 [Train]:  38%|███▊      | 5/13 [07:19<09:48, 73.56s/it, mse=0.198235]

In [6]:
import os

print("Running diagnostic on the dataset structure...\n")

KAGGLE_DATA_ROOT = "/kaggle/input/ts-satfire/ts-satfire"
all_dirs = sorted([
    os.path.join(KAGGLE_DATA_ROOT, d) for d in os.listdir(KAGGLE_DATA_ROOT) 
    if os.path.isdir(os.path.join(KAGGLE_DATA_ROOT, d))
])

if len(all_dirs) == 0:
    print("Error: No directories found in the root path.")
else:
    sample_dir = all_dirs[0]
    print(f"Inspecting directory: {sample_dir}")
    print("-" * 50)
    
    for item in os.listdir(sample_dir):
        item_path = os.path.join(sample_dir, item)
        if os.path.isdir(item_path):
            print(f"[Directory] {item}")
            files = os.listdir(item_path)
            print(f"  -> Contains {len(files)} files. First few: {files[:3]}")
        else:
            print(f"[File] {item}")
    print("-" * 50)

Running diagnostic on the dataset structure...

Inspecting directory: /kaggle/input/ts-satfire/ts-satfire/20562846
--------------------------------------------------
[Directory] VIIRS_Night
  -> Contains 8 files. First few: ['2017-04-25_VIIRS_Night.tif', '2017-04-29_VIIRS_Night.tif', '2017-04-27_VIIRS_Night.tif']
[Directory] FirePred
  -> Contains 8 files. First few: ['2017-04-28_FirePred.tif', '2017-04-30_FirePred.tif', '2017-04-24_FirePred.tif']
[Directory] VIIRS_Day
  -> Contains 8 files. First few: ['2017-04-25_VIIRS_Day.tif', '2017-04-23_VIIRS_Day.tif', '2017-04-26_VIIRS_Day.tif']
[Directory] ESRI_LULC
  -> Contains 1 files. First few: ['2017-01-01_ESRI_LULC.tif']
--------------------------------------------------
